In [1]:
import os
import json
import urllib.request
import importlib, pathlib, sys
from urllib.parse import urlparse
from datetime import datetime, timezone

import teehr
import pandas as pd
from teehr.evaluation.spark_session_utils import create_spark_session

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf
from teehr import Bootstrappers as bs

from teehr.models.filters import TableFilter

from pyspark.sql import functions as F

from pyspark.sql import DataFrame

import copy
import time

teehr.__version__

'0.7.0'

In [2]:


BRANCH = "improve-nwmd-preprocessing-efficiency"
MOD_DIR = pathlib.Path("/home/jovyan/nwmd_modules")
MOD_DIR.mkdir(parents=True, exist_ok=True)

RAW = ("https://raw.githubusercontent.com/RTIInternational/teehr-hub/"
       f"{BRANCH}/warehouse/remote/03_preprocessing/nwm_diagnostics/utils.py")

# cache-buster: raw.githubusercontent caches branch URLs for ~5 minutes
urllib.request.urlretrieve(f"{RAW}?t={time.time()}", MOD_DIR / "utils.py")

sys.path.insert(0, str(MOD_DIR))
import utils
importlib.reload(utils)  # picks up a re-download without a kernel restart
print("loaded", utils.__file__)


loaded /home/jovyan/nwmd_modules/utils.py


In [3]:
pod_template_path = utils.create_ondemand_pod_template()

Wrote alternate pod template to /home/jovyan/executor-pod-template-ondemand.yaml


In [4]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

# ---------------------------------------------------------------------------
# Declarative dimension spec
# ---------------------------------------------------------------------------
# Every group-by column of the output table is declared once, below, and the
# calculated fields, row expansions, group_by lists, nullable_fields and
# partition_by are all *derived* from those declarations. Adding a dimension
# (e.g. water_year) is a single spec entry rather than matching edits in eight
# hand-maintained places.

# Columns of fcst_joined_timeseries that do NOT identify a timeseries.
NON_UNIQUE_FIELDS = [
    "primary_value", "secondary_value", "created_at", "updated_at", "value_time",
]
# Identifies a single forecast, so it must be a key of the bin aggregation, but
# is aggregated away before the final metrics.
BIN_ONLY_FIELDS = ["reference_time"]
# Nullable in the output regardless of the dimension specs.
ALWAYS_NULLABLE = ["member"]
# Excluded when deciding which rows form one timeseries for percentile events.
REMOVE_FOR_QUANTILES = ["secondary_location_id", "reference_time", "member"]

# teehr builds MERGE partition filters by running SELECT DISTINCT / MIN-MAX over
# the *lazy* source view, which executes the entire bootstrap DAG once before the
# MERGE executes it again -- 2x the most expensive stage in the pipeline, to prune
# partitions we gain little from pruning. Keeping them off is also what makes a
# nullable partition column safe (see DimensionSpec.nullable_partition_fields).
USE_PARTITION_FILTERS = False

# Dimension stages. The distinction matters for both correctness and cost:
PRE_BIN = "pre_bin"    # levels select row SUBSETS -> must expand before the bin agg
BIN = "bin"            # plain grouping key, no expansion
POST_BIN = "post_bin"  # levels keep all rows -> expand after the bin agg


@dataclass(frozen=True)
class Level:
    """One level of a dimension.

    values : one SQL expression per name in Dimension.names.
    keep : boolean SQL; rows where this is false are dropped for this level.
    payload : extra output column -> source column, for dimensions that pivot
        metric *outputs* into rows (window_agg) rather than replicating rows.
    """

    values: Tuple[str, ...]
    keep: str = "true"
    payload: Dict[str, str] = field(default_factory=dict)


@dataclass(frozen=True)
class Dimension:
    """One or more output columns expanded together as correlated levels.

    names is a tuple rather than a str so a single dimension can express
    *correlated* levels -- Spark GROUPING SETS emulated by row replication,
    which teehr's flat group_by cannot express natively. Only length-1 tuples
    are needed today; see the water_year notes below for when that changes.
    """

    names: Tuple[str, ...]
    stage: str
    levels: Tuple[Level, ...] = ()
    calculated_fields: Tuple = ()       # teehr CFs that materialize the source columns
    consumes: Tuple[str, ...] = ()      # helper columns dropped after the stack
    payload_fields: Tuple[str, ...] = ()
    nullable_names: Tuple[str, ...] = ()
    partition_names: Tuple[str, ...] = ()
    in_bin_group: bool = True           # name(s) are a key of the bin aggregation
    in_final_group: bool = True         # name(s) are a key of the final aggregation

    def validate(self):
        for lvl in self.levels:
            assert len(lvl.values) == len(self.names), (
                f"{self.names}: level has {len(lvl.values)} values, "
                f"expected {len(self.names)}"
            )
            assert tuple(sorted(lvl.payload)) == tuple(sorted(self.payload_fields)), (
                f"{self.names}: every level must emit the same payload keys"
            )


@dataclass
class DimensionSpec:
    """The dimensions of one output table, plus every list derived from them."""

    entity_fields: List[str]
    dims: List[Dimension]
    # Entity columns to partition on, in addition to any dimension that sets
    # partition_names. configuration_name is always non-null, low cardinality,
    # in group_by (so it reaches the MERGE ON clause and Iceberg can prune on
    # it), and each run writes exactly one -- so a run touches one partition.
    entity_partition_fields: Tuple[str, ...] = ("configuration_name",)

    def __post_init__(self):
        for dim in self.dims:
            dim.validate()
        names = [n for d in self.dims for n in d.names]
        assert len(set(names)) == len(names), f"duplicate dimension names: {names}"
        assert not (set(names) & set(self.entity_fields)), (
            f"dimension name collides with a joined-timeseries column: "
            f"{set(names) & set(self.entity_fields)}"
        )
        assert set(self.entity_partition_fields) <= set(self.entity_fields), (
            f"unknown entity partition field(s): "
            f"{set(self.entity_partition_fields) - set(self.entity_fields)}"
        )

    def at(self, stage) -> List[Dimension]:
        return [d for d in self.dims if d.stage == stage]

    @property
    def calculated_fields(self) -> List:
        return [cf for d in self.dims for cf in d.calculated_fields]

    @property
    def group_by_bin(self) -> List[str]:
        """Keys of the per-forecast / per-lead-time-bin aggregation."""
        return list(self.entity_fields) + [
            n for d in self.dims if d.in_bin_group for n in d.names
        ]

    @property
    def group_by(self) -> List[str]:
        """Keys of the final metric aggregation (drops reference_time)."""
        return [f for f in self.entity_fields if f not in BIN_ONLY_FIELDS] + [
            n for d in self.dims if d.in_final_group for n in d.names
        ]

    @property
    def nullable_fields(self) -> List[str]:
        return list(ALWAYS_NULLABLE) + [
            n for d in self.dims for n in d.nullable_names
        ]

    @property
    def partition_by(self) -> List[str]:
        return list(self.entity_partition_fields) + [
            n for d in self.dims for n in d.partition_names
        ]

    @property
    def nullable_partition_fields(self) -> List[str]:
        """Partition columns that can be NULL.

        Iceberg itself is perfectly happy with these -- a NULL identity-partition
        value simply gets its own partition. The constraint is teehr's, and only
        when MERGE partition filters are enabled: _build_partition_filters builds
        `t.<f> IN (...)` for string columns (from SELECT DISTINCT ... WHERE <f>
        IS NOT NULL) and `t.<f> >= min AND t.<f> <= max` for numeric ones, and
        BOTH evaluate to NULL -- i.e. not matched -- for a NULL partition value.
        Those target rows would then fall outside the merge and be re-INSERTed,
        and therefore duplicated, on every upsert. Harmless while
        USE_PARTITION_FILTERS is False; asserted at the write site if it is not.
        """
        nullable = set(self.nullable_fields)
        return [f for f in self.partition_by if f in nullable]


def expand_dimension(tbl, dim: Dimension):
    """Expand one Dimension's levels into rows, materializing dim.names.

    Generates the stack() SQL from the spec, so its arity and its level list can
    never drift apart. Replaces both hand-written stack() strings.
    """
    if not dim.levels:
        return tbl  # plain grouping key, nothing to expand

    out_names = [*dim.names, "_keep_row", *dim.payload_fields]
    rows = [
        [*lvl.values, lvl.keep, *(lvl.payload[p] for p in dim.payload_fields)]
        for lvl in dim.levels
    ]
    # The stack() body may still reference columns that are not selected as base
    # columns (above_q85, quarter, mean_primary_value, ...); that is exactly how
    # the helper columns get consumed and dropped in one step.
    dropped = set(dim.consumes) | set(out_names)
    base_cols = [c for c in tbl.to_sdf().columns if c not in dropped]
    body = ",\n            ".join(", ".join(r) for r in rows)

    stacked = tbl.selectExpr(
        *base_cols,
        f"""
        stack(
            {len(rows)},
            {body}
        ) as ({", ".join(out_names)})
        """,
    )
    if any(lvl.keep != "true" for lvl in dim.levels):
        stacked = stacked.where("_keep_row")
    return stacked.select(*base_cols, *dim.names, *dim.payload_fields)


# --- The dimensions of nwmd_metrics_by_location ----------------------------

WINDOW_AGGS = (("mean", s.Average), ("min", s.Minimum), ("max", s.Maximum))
WINDOW_VALUE_FIELDS = ("primary_value", "secondary_value")
BIN_COUNT_FIELD = "n_in_bin"


def window_metrics() -> List:
    """Per-bin aggregations: one Signature metric per (window agg, value field)."""
    metrics = [
        cls(primary_field_name=fld, output_field_name=f"{label}_{fld}")
        for label, cls in WINDOW_AGGS
        for fld in WINDOW_VALUE_FIELDS
    ]
    metrics.append(
        s.Count(primary_field_name="secondary_value", output_field_name=BIN_COUNT_FIELD)
    )
    return metrics


def window_agg_dimension() -> Dimension:
    """Pivots the per-bin mean/min/max columns into rows keyed by window_agg.

    This is a third shape: it pivots metric *outputs* into rows rather than
    replicating input rows, which is what `payload` expresses. It is still a
    Dimension because window_agg genuinely is part of the output key.
    """
    return Dimension(
        names=("window_agg",),
        stage=POST_BIN,
        in_bin_group=False,  # does not exist until after the bin aggregation
        payload_fields=WINDOW_VALUE_FIELDS,
        consumes=tuple(
            f"{label}_{fld}" for label, _ in WINDOW_AGGS for fld in WINDOW_VALUE_FIELDS
        ),
        levels=tuple(
            Level(
                values=(f"'{label}'",),
                payload={fld: f"{label}_{fld}" for fld in WINDOW_VALUE_FIELDS},
            )
            for label, _ in WINDOW_AGGS
        ),
    )


def threshold_dimension(quantiles, quantile_group) -> Dimension:
    """Above-percentile event levels, plus a NULL level meaning "all rows".

    PRE_BIN: each level selects a subset of rows, so the per-bin mean/min/max
    must be computed over that level's rows only.

    Note: both the threshold and the event detection are based on primary_value.
    This may differ from the way it is done in the NWM Explorer.
    """
    def event_col(q):
        """The one place a quantile maps to its event-flag column name."""
        return f"above_q{int(q * 100)}"

    cols = tuple(event_col(q) for q in quantiles)
    return Dimension(
        names=("threshold",),
        stage=PRE_BIN,
        nullable_names=("threshold",),
        consumes=cols,
        calculated_fields=tuple(
            tcf.AbovePercentileEventDetection(
                quantile=q,
                output_event_field_name=event_col(q),
                skip_event_id=True,
                value_field_name="primary_value",
                uniqueness_fields=quantile_group,
            )
            for q in quantiles
        ),
        levels=(
            Level(values=("cast(null as string)",)),  # all rows, no threshold
            *(Level(values=(f"'{col}'",), keep=col) for col in cols),
        ),
    )


# Temporal rollups, as grouping sets over (water_year, quarter). Selecting a
# rollup adds one level, i.e. one more copy of every post-bin row -- and the
# post-bin stream is what feeds the bootstrap, the most expensive stage in the
# pipeline. A NULL in a position means that column is aggregated ACROSS for
# that level.
#
#   quarter    -> one row per quarter          (water_year, quarter)
#   water_year -> one row per water year       (water_year, NULL)
#   all        -> one row for the whole run    (NULL,       NULL)
#
# IMPORTANT: a rollup covers the data THIS RUN read, not everything in the
# table. Running one water year at a time and upserting would overwrite the
# "all" row with just that run's data, and the per-water-year rows already in
# the table cannot be combined into it after the fact (NSE, KGE and correlation
# are not averageable, and the bin-level rows they would need are not
# persisted). So enable "all" only on a run whose reference_time filters span
# the whole period of record.
TEMPORAL_ROLLUPS = {
    "quarter":    ("cast(water_year as int)", "cast(quarter as string)"),
    "water_year": ("cast(water_year as int)", "cast(null as string)"),
    "all":        ("cast(null as int)",       "cast(null as string)"),
}
DEFAULT_ROLLUPS = ("quarter", "water_year")
# The "collapsed" value per position, used to derive which columns are nullable.
TEMPORAL_NULL_SQL = ("cast(null as int)", "cast(null as string)")


def temporal_dimension(rollups) -> Dimension:
    """water_year + quarter expanded together, as one set of correlated levels.

    They have to be one Dimension rather than two: because every quarter belongs
    to exactly one water year, an independent NULL level on water_year would
    just duplicate the per-quarter rows. A rollup is only meaningful when the
    finer column collapses with it -- which is what a grouping set expresses.
    """
    names = ("water_year", "quarter")
    unknown = [r for r in rollups if r not in TEMPORAL_ROLLUPS]
    assert not unknown, (
        f"unknown temporal rollup(s) {unknown}; choose from {list(TEMPORAL_ROLLUPS)}"
    )
    assert rollups, "at least one temporal rollup is required"
    # Canonical order, however the config happened to list them.
    values = [TEMPORAL_ROLLUPS[r] for r in TEMPORAL_ROLLUPS if r in rollups]
    nullable = tuple(
        name for i, name in enumerate(names)
        if any(v[i] == TEMPORAL_NULL_SQL[i] for v in values)
    )
    return Dimension(
        names=names,
        stage=POST_BIN,
        nullable_names=nullable,
        # Stays a partition key even when the "all" rollup makes it nullable:
        # the NULL rollup rows just land in their own Iceberg partition. See
        # DimensionSpec.nullable_partition_fields for the one caveat.
        partition_names=("water_year",),
        calculated_fields=(
            rcf.WaterYear(
                input_field_name="reference_time",  # not the "value_time" default
                output_field_name="water_year",
            ),
            rcf.GenericSQL(
                output_field_name="quarter",
                sql_statement=(
                    "CONCAT(YEAR(reference_time), '-Q', QUARTER(reference_time))"
                ),
            ),
        ),
        levels=tuple(Level(values=v) for v in values),
    )


def build_dimensions(config, quantile_group) -> List[Dimension]:
    """Declare every derived dimension of the output table."""
    return [
        temporal_dimension(config.get("rollups", DEFAULT_ROLLUPS)),
        Dimension(
            names=("forecast_lead_time_bin",),
            stage=BIN,
            calculated_fields=(
                rcf.ForecastLeadTimeBins(
                    bin_size=pd.Timedelta(
                        hours=config.get("forecast_lead_time_bin_hours")
                    ),
                    output_field_name="forecast_lead_time_bin",
                ),
            ),
        ),
        threshold_dimension((0.85, 0.95, 0.99), quantile_group),
        window_agg_dimension(),
    ]


# --- Metrics ---------------------------------------------------------------

# Each entry yields BOTH a point estimate and its bootstrap CI, from the same
# kwargs. Previously the *_boot variants silently omitted add_epsilon, so the
# interval described a different estimator than the point value it accompanied
# and the point value could fall outside its own CI.
BOOTSTRAPPED_METRICS = (
    (dm.RelativeMean, "relative_mean", {}),
    (dm.RelativeMedian, "relative_median", {}),
    (dm.RelativeMinimum, "relative_minimum", {}),
    (dm.RelativeMaximum, "relative_maximum", {}),
    (dm.RelativeStandardDeviation, "relative_standard_deviation", {}),
    (dm.RelativeBias, "relative_bias", {"add_epsilon": True}),
    (dm.NashSutcliffeEfficiency, "nash_sutcliffe_efficiency", {"add_epsilon": True}),
    (dm.KlingGuptaEfficiency, "kling_gupta_efficiency", {"add_epsilon": True}),
    (dm.PearsonCorrelation, "pearson_correlation", {"add_epsilon": True}),
)


def build_metrics(bootstrap) -> List:
    """Final metrics: signatures, point estimates, and matching bootstrap CIs."""
    metrics = [
        s.Count(),
        s.Average(),
        s.Minimum(),
        s.Maximum(),
        # Timesteps behind each bin value, carried through the window_agg pivot.
        s.Sum(primary_field_name=BIN_COUNT_FIELD, output_field_name="n_timesteps"),
    ]
    for cls, name, kwargs in BOOTSTRAPPED_METRICS:
        metrics.append(cls(**kwargs))
        # unpack_results=True is safe as of the pinned teehr commit:
        # post_process_metric_results now derives the quantile keys statically
        # via derive_map_key_list(), so unpacking no longer triggers a
        # per-metric .first() Spark action that re-ran the entire upstream DAG
        # (the confirmed cause of the "ShuffleMapStage ... first at
        # teehr/querying/utils.py:207" crashes recorded in profiling.md).
        metrics.append(
            cls(
                output_field_name=f"{name}_boot",
                bootstrap=bootstrap,
                unpack_results=True,
                **kwargs,
            )
        )
    return metrics


def generate_nwmd_metrics(spark, config, output_table_name):
    """Generate the teehr.nwmd_metrics_by_location table for the given config.

    config format:
        {
            "configurations": ["nwm30_medium_range"],
            "forecast_lead_time_bin_hours": 24,
            "start_reference_time": "2025-10-01T00:00",
            "end_reference_time": "2026-10-01T00:00",
            # optional; defaults to ("quarter", "water_year").
            # Add "all" for a period-of-record rollup -- see TEMPORAL_ROLLUPS
            # for what that costs and when it is valid.
            "rollups": ["quarter", "water_year"]
        },

    Args:
        spark (SparkSession): The Spark session to use for processing.
        config (dict): Configuration dictionary containing necessary parameters.
        output_table_name (str): Name of the output table to store the generated metrics.
    """

    configurations = config.get("configurations")
    start_reference_time = config.get("start_reference_time")
    end_reference_time = config.get("end_reference_time")

    start = time.perf_counter()

    ev = teehr.RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

    joined_cols = ev.table("fcst_joined_timeseries").to_sdf().columns
    entity_fields = [c for c in joined_cols if c not in NON_UNIQUE_FIELDS]
    quantile_group = [c for c in entity_fields if c not in REMOVE_FOR_QUANTILES]
    print(f"Entity (uniqueness) fields: {entity_fields}")

    spec = DimensionSpec(
        entity_fields=entity_fields,
        dims=build_dimensions(config, quantile_group),
    )
    print(f"Dimensions: {[n for d in spec.dims for n in d.names]}")
    print(f"Final group_by: {spec.group_by}")

    filters = [
        TableFilter(
            column="configuration_name",
            operator="in",
            value=configurations
        ),
        TableFilter(
            column="reference_time",
            operator=">=",
            value=start_reference_time,
        ),
        TableFilter(
            column="reference_time",
            operator="<",
            value=end_reference_time,
        )
    ]

    # temporarily sample a subset of location IDs for testing purposes
    # ids = ev.locations.filter("id like 'usgs-%'").to_sdf().select("id")
    # sample = ids.sample(False, 0.5, seed=456).limit(10).collect()
    # location_ids = [r.id for r in sample]
    # print("Number of location_ids:", len(location_ids))
    # filters.append(
    #     TableFilter(
    #         column="primary_location_id",
    #         operator="in",
    #         value=location_ids
    #     )
    # )

    # Get raw joined timeseries with every dimension's calculated fields applied.
    tbl = (
        ev.table("fcst_joined_timeseries")
        .filter(filters)
        .add_calculated_fields(spec.calculated_fields)
    )

    # Row-filtering dimensions expand BEFORE the bin aggregation: each level
    # selects a subset of rows, so the per-bin mean/min/max must see only that
    # level's rows.
    for dim in spec.at(PRE_BIN):
        tbl = expand_dimension(tbl, dim)

    bin_aggs_tbl = tbl.aggregate(
        group_by=spec.group_by_bin,
        metrics=window_metrics(),
    )

    # Rollup dimensions expand AFTER it. Every level keeps all rows, and each is
    # functionally determined by reference_time (itself a bin key), so the result
    # is identical to expanding pre-bin -- but without multiplying the scan, the
    # event detection and the bin-agg shuffle, which handle far more rows than
    # this post-bin stream does.
    for dim in spec.at(POST_BIN):
        bin_aggs_tbl = expand_dimension(bin_aggs_tbl, dim)

    # Configure bootstrap
    bootstrap = bs.Stationary(
        reps=1000,
        seed=1234,
        quantiles=[0.025, 0.975]
    )

    group_by = spec.group_by

    results = (
        bin_aggs_tbl.aggregate(
            group_by=group_by,
            metrics=build_metrics(bootstrap),
        )
        .order_by(group_by)
        .add_geometry()
    )

    # print(results.explain(mode="simple"))

    full_table_name = f"iceberg.teehr.{output_table_name}"

    if ev.spark.catalog.tableExists(full_table_name):
        assert not (USE_PARTITION_FILTERS and spec.nullable_partition_fields), (
            f"partition filters are enabled and {spec.nullable_partition_fields} "
            f"can be NULL; those rows would go unmatched by the MERGE and be "
            f"duplicated on every upsert"
        )
        results.write_to(
            table_name=output_table_name,
            write_mode="upsert",
            # The FULL key, with nullable_fields naming the columns that need
            # null-safe (<=>) matching. _build_on_clause only applies <=> to
            # fields present in BOTH lists, so the previous
            # "group_by minus nullables" left threshold and member out of the
            # MERGE ON clause entirely -- one target row then matched every
            # threshold level, and threshold landed in the UPDATE SET clause.
            uniqueness_fields=group_by,
            nullable_fields=spec.nullable_fields,
            use_partition_filters=USE_PARTITION_FILTERS,
        )
    else:
        results.write_to(
            table_name=output_table_name,
            write_mode="create_or_replace",
            partition_by=spec.partition_by,
        )

    # Read the metric column names off the written result rather than off the metric
    # models. With unpack_results=True each bootstrap metric's MapType column is
    # replaced by one column per quantile (e.g. relative_mean_boot ->
    # relative_mean_boot_0_025, _0_975), so `metric.output_field_name` would
    # advertise columns that don't exist in the table. `.columns` is schema-only,
    # so this costs no Spark action.
    # "name" and "geometry" come from add_geometry(), not from a metric.
    non_metric_columns = set(group_by) | {"name", "geometry"}
    metric_columns = [
        c for c in results.to_sdf().columns if c not in non_metric_columns
    ]

    properties = {
        "description": "NWM diagnostics metrics by location ID",
        "group_by": ", ".join(group_by),
        "metrics": ", ".join(metric_columns)
    }

    for key, value in properties.items():
        ev.spark.sql(f"""
            ALTER TABLE {full_table_name} SET TBLPROPERTIES ('{key}' = '{value}')
        """)

    end = time.perf_counter()

    elapsed_seconds = end - start
    print(f"{elapsed_seconds:.6f} s")

    # Capture resource-usage metrics for this run BEFORE spark.stop() (the REST API
    # stops responding once the session ends). Paste the printed markdown row into
    # the Profiling table above to keep a running record.
    # n_locations = len(location_ids) if "location_ids" in globals() else "All"
    # n_days = utils._infer_days_from_filters(filters)

    # run_metrics = utils.capture_spark_run_metrics(spark, label=f"{n_locations} locations, {n_days} days")
    # utils.report_utilization(run_metrics, elapsed_seconds)

    # print(
    #     f"\n| {n_locations} | {n_days} | {bootstrap.reps} | Cluster - {utils.spark_config_summary(spark)} "
    #     f"| {elapsed_seconds:.0f}s | (see run_metrics above) |"
    # )

    # Run this against the still-live session (spark.stop() is commented out below)
    # to see the actual failure reason for the retried/failed stages from this run,
    # without needing to read it off the Spark UI by hand.
    # stage_failures = utils.get_stage_attempt_failures(spark)

    # spark.stop()

In [5]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=64,
    executor_memory="16g",
    executor_cores=2,
    aws_profile="default",
    pod_template_path=pod_template_path,
    update_configs={
        "spark.sql.shuffle.partitions": 1024,
        "spark.sql.adaptive.coalescePartitions.enabled": "false",
        "spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict": "false",
        "spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE": "vectorized",
        "spark.executor.memoryOverhead": "4g",
    }
)

# spark = create_spark_session()

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:📦 Configuring Spark cluster with container image: None
INFO:teehr.evaluation.spark_session_utils:🔍 Initial spark namespace from ENV: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔍 Connecting to Kubernetes API: https://172.20.0.1:443
INFO:teehr.evaluation.spark_session_utils:🎯 Executor namespace: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔐 Executor service account: spark (in teehr-hub)
INFO:teehr.evaluation.spark_session_utils:🔐 Using in-cluster authentication
INFO:teehr.evaluation.spark_session_utils:🔗 Setting driver host to pod IP: 10.0.3.199
INFO:teehr.evaluation.spark_session_utils:✅ Spark cluster configuration successful!
INFO:teehr.evaluation.spark_session_utils:   - Executor instances: 64
INFO:teehr.evaluation.spark_session_utils:   - Executor memory: 16g
INFO:teehr.evaluation.spark_session_utils:   - Executor cores: 2
INFO:teehr.evaluati

In [6]:
spark.sql("DROP TABLE IF EXISTS nwmd_metrics_by_location_test PURGE")
spark.sql("DROP TABLE IF EXISTS nwmd_metrics_by_location_v2 PURGE")

DataFrame[]

In [7]:
# "rollups" is optional and defaults to ("quarter", "water_year").
# Add "all" for a period-of-record row (water_year IS NULL AND quarter IS NULL),
# but only on a run whose reference_time range covers the whole record -- a
# rollup summarizes what THIS run read, and cannot be assembled from separate
# per-water-year runs afterwards. It also adds a third copy of every post-bin
# row, i.e. ~1.5x the bootstrap work. Drop to ["quarter"] for the cheapest run.
configurations = [
    {
        "configurations": ["nwm30_medium_range"],
        "forecast_lead_time_bin_hours": 24,
        "start_reference_time": "2024-10-01T00:00",
        "end_reference_time": "2026-10-01T00:00"
    },
    {
        "configurations": ["nwm30_short_range"],
        "forecast_lead_time_bin_hours": 6,
        "start_reference_time": "2024-10-01T00:00",
        "end_reference_time": "2026-10-01T00:00"
    }
]

In [8]:
for config in configurations:
    print(config)
    generate_nwmd_metrics(
        spark, 
        config, 
        output_table_name = "nwmd_metrics_by_location_v2"
    )

INFO:teehr.evaluation.evaluation:Using provided Spark session.


{'configurations': ['nwm30_medium_range'], 'forecast_lead_time_bin_hours': 24, 'start_reference_time': '2024-10-01T00:00', 'end_reference_time': '2026-10-01T00:00'}


INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.
INFO:teehr.evaluation.tables.generic_table:Getting table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: fcst_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.fcst_joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Setting filter [TableFilter(column='configuration_name', operator=<FilterOperators.isin: 'in'>, value=['nwm30_medium_range']), TableFilter(column='reference_time', operator=<FilterOpera

Entity (uniqueness) fields: ['reference_time', 'primary_location_id', 'secondary_location_id', 'configuration_name', 'unit_name', 'variable_name', 'member']
Dimensions: ['water_year', 'quarter', 'forecast_lead_time_bin', 'threshold', 'window_agg']
Final group_by: ['primary_location_id', 'secondary_location_id', 'configuration_name', 'unit_name', 'variable_name', 'member', 'water_year', 'quarter', 'forecast_lead_time_bin', 'threshold', 'window_agg']


INFO:teehr.evaluation.dataframe_base:Performing the aggregation.
INFO:teehr.evaluation.dataframe_base:Performing the aggregation.
INFO:teehr.evaluation.dataframe_base:Setting order_by ['primary_location_id', 'secondary_location_id', 'configuration_name', 'unit_name', 'variable_name', 'member', 'water_year', 'quarter', 'forecast_lead_time_bin', 'threshold', 'window_agg'].
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations.
INFO:teehr.evaluation.dataframe_base:Writing to table: nwmd_metrics_by_location_v2.
INFO:teehr.evaluation.tables.generic_table:Getting table: nwmd_metrics_by_location_v2.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: nwmd_metrics_by_location_v2.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.nwmd_metrics_by_location_v2.
INFO:teehr.evaluat

Py4JJavaError: An error occurred while calling o175.sql.
: org.apache.spark.SparkException: Job aborted due to stage failure: ShuffleMapStage 22 ($anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768) has failed the maximum allowable number of times: 4. Most recent failure reason:
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:439)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1253)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:983)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:87)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$10.nextCur(Iterator.scala:594)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:608)
	at scala.collection.Iterator$$ano

In [ ]:
spark.sql("USE iceberg.teehr")

DataFrame[]

In [ ]:
spark.sql("""
SELECT *
FROM nwmd_metrics_by_location_v2 
LIMIT 10
""").show()

+-------------------+---------------------+------------------+---------+--------------------+------+----------+-------+----------------------+---------+----------+-----+------------------+--------------------+-----------------+-----------+------------------+-------------------+--------------------+------------------+---------------------------+--------------------+-------------------------+----------------------+-------------------+------------------------+------------------------+--------------------------+--------------------------+---------------------------+---------------------------+---------------------------+---------------------------+--------------------------------------+--------------------------------------+------------------------+------------------------+------------------------------------+------------------------------------+---------------------------------+---------------------------------+------------------------------+------------------------------+----------------

In [ ]:
spark.sql("""
SELECT 
    configuration_name,
    collect_set(forecast_lead_time_bin) as forecast_lead_time_bins,
    collect_set(water_year) as water_years,
    collect_set(threshold) as thresholds,
    collect_set(quarter) as quarters,
    collect_set(window_agg) as window_aggs
FROM nwmd_metrics_by_location_v2 
GROUP BY configuration_name
""").show(truncate=False)

+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------------------------+------------------------------------------------------------------------+----------------+
|configuration_name|forecast_lead_time_bins                                                                                                                                               |water_years |thresholds                       |quarters                                                                |window_aggs     |
+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------------------------+------------------------------------------------------------------------+----------------+
|nwm30_medium_range|[P1DT

In [9]:
spark.stop()